# Multi-Host Distributed PyTorch Training with Shared ReadWriteMany (Filestore) Volume on GKE

This notebook demonstrates how to launch a **multi-host distributed PyTorch training job** across a multi-host Cloud TPU slice (e.g. 2 hosts × 4 chips = 8 TPU v5e chips) using a **shared `ReadWriteMany` (Filestore) volume** instead of a `ConfigMap` — eliminating the 1 MiB `ConfigMap` limit and avoiding Docker image rebuilds when training large repositories.

---

## How Shared `ReadWriteMany` Volume Training Works on GKE

When your codebase exceeds the 1 MiB Kubernetes `ConfigMap` limit (or contains nested Python packages, YAML configs, tokenizers, and assets), both your **Workspace pod** and all **TPU `TrainJob` worker pods** mount the same `ReadWriteMany` (`shared-workspace-rwx`) Filestore PVC:

1. **Per-Workspace Isolation (`subPathExpr`)**: The `jupyterlab` `WorkspaceKind` mounts `shared-workspace-rwx` at `/home/jovyan/shared` with `subPathExpr: workspaces/$(WORKSPACE_NAME)`. Each workspace gets its own isolated directory on the shared NFS volume (`workspaces/<workspace-name>/`).
2. **Live VS Code Sync (`jupyter-workspace-sync`)**: Setting `"jupyterSync.remoteBaseDir": "shared/${workspaceFolderBasename}"` in VS Code syncs your local Git repository directly into `/home/jovyan/shared/<repo-name>/` (`workspaces/<workspace-name>/<repo-name>/` on the PVC). Every `Ctrl+S` save updates the shared volume in ~300 ms.
3. **Zero-Copy Multi-Host Mount (`submit_multihost_training`)**: `tpu_trainer.py` automatically resolves the PVC `subPath` for `src_dir` (`workspaces/<workspace-name>/<repo-name>/examples/torch_tpu/src`) and mounts it at `/workspace` on every TPU host pod.

```mermaid
flowchart TD
    subgraph Laptop["Local Laptop (Optional)"]
      VSC["VS Code (jupyter-workspace-sync)<br/>jupyterSync.remoteBaseDir: shared/&#36;{workspaceFolderBasename}"]
    end

    subgraph Notebook["Interactive Workspace Pod"]
      NB["JupyterLab / Kernel<br/>/home/jovyan/shared (subPathExpr: workspaces/&#36;(WORKSPACE_NAME))"]
    end

    subgraph Storage["Cloud Filestore RWX (1 TiB)"]
      PVC[("PVC: shared-workspace-rwx<br/>subPath: workspaces/&lt;ws-name&gt;/.../src")]
    end

    subgraph Node0["TPU Host 0 (trainer-node-0-0)"]
      TRUN0["torchrun --nproc_per_node=4<br/>Mounts PVC subPath at /workspace"]
      R0["Ranks 0..3 (4 TPU v5e Cores)"]
    end

    subgraph Node1["TPU Host 1 (trainer-node-0-1)"]
      TRUN1["torchrun --nproc_per_node=4<br/>Mounts PVC subPath at /workspace"]
      R4["Ranks 4..7 (4 TPU v5e Cores)"]
    end

    VSC -->|"~300ms delta sync"| NB
    NB <-->|"Read/Write code & checkpoints"| PVC
    PVC ===>|"ReadWriteMany mount"| TRUN0 & TRUN1
    TRUN0 --> R0
    TRUN1 --> R4
    R0 <== "ICI Network (tpu_dist AllReduce)" ==> R4
```

## 1. Setup & Shared RWX Volume Verification

Discover the current Kubernetes namespace and verify that the shared `ReadWriteMany` PVC (`shared-workspace-rwx`) and local mount (`/home/jovyan/shared`) are ready.

In [ ]:
import os
import pathlib
import sys
import time
from kubernetes import client as k8s_client, config as k8s_config

# Load in-cluster or local kubeconfig
try:
    k8s_config.load_incluster_config()
except Exception:
    k8s_config.load_kube_config()

def get_current_namespace() -> str:
    if os.environ.get("TENANT_NAMESPACE"):
        return os.environ["TENANT_NAMESPACE"]
    ns_path = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
    if os.path.exists(ns_path):
        with open(ns_path) as f:
            return f.read().strip()
    return "default"

namespace = get_current_namespace()
shared_pvc_name = os.environ.get("SHARED_RWX_PVC", "shared-workspace-rwx")
shared_mount_path = pathlib.Path(os.environ.get("SHARED_RWX_MOUNT", "/home/jovyan/shared"))

print(f"Target Kubernetes namespace : {namespace}")
print(f"Working directory (cwd)     : {pathlib.Path.cwd()}")
print(f"Shared RWX mount path       : {shared_mount_path} (exists={shared_mount_path.exists()})")

core_v1 = k8s_client.CoreV1Api()
try:
    pvc = core_v1.read_namespaced_persistent_volume_claim(name=shared_pvc_name, namespace=namespace)
    print(f"Shared RWX PVC              : {pvc.metadata.name} (status={pvc.status.phase}, accessModes={pvc.spec.access_modes})")
except Exception as e:
    print(f"Warning: Could not read PVC '{shared_pvc_name}' in namespace '{namespace}': {e}")

## 2. Inspect Project File Tree (`src/` & `tpu_trainer.py`)

With a shared `ReadWriteMany` volume synced via `jupyter-workspace-sync` (`"jupyterSync.remoteBaseDir": "shared/${workspaceFolderBasename}"`), your multi-file project tree lives directly on the shared Filestore volume at `/home/jovyan/shared/<repo-name>/examples/torch_tpu/` — no inline `%%writefile` cells or 1 MiB `ConfigMap` packaging needed:

```text
examples/torch_tpu/
├── torch_tpu_multihost_rwx.ipynb   # Orchestrator notebook
├── tpu_trainer.py                  # Kubeflow Trainer v2 + RWX PVC mount helper
└── src/
    ├── models.py                   # Neural network architecture (MLPClassifier)
    ├── dataset.py                  # Synthetic data generation & DistributedSampler
    └── train.py                    # Multi-host torchrun DDP training entrypoint
```

Edit any file in `src/` or `tpu_trainer.py` locally in VS Code (or in JupyterLab) and re-run Section 3 to immediately train with your updated code across all TPU hosts.

In [ ]:
import pathlib

project_dir = pathlib.Path.cwd()
src_dir = project_dir / "src"

assert (project_dir / "tpu_trainer.py").exists(), f"Missing tpu_trainer.py in {project_dir}"
assert src_dir.is_dir(), f"Missing src/ directory in {project_dir}"

print(f"Project directory: {project_dir}\n")
print("Project files on disk:")
print(f"├── tpu_trainer.py ({(project_dir / 'tpu_trainer.py').stat().st_size} bytes)")
print("└── src/")
py_files = sorted(src_dir.glob("*.py"))
for idx, f in enumerate(py_files):
    branch = "└──" if idx == len(py_files) - 1 else "├──"
    print(f"    {branch} {f.name} ({f.stat().st_size} bytes)")

## 3. Submit Multi-Host Distributed Training with Shared Volume (`submit_multihost_training`)

All Kubernetes plumbing (shared `ReadWriteMany` PVC `subPath` resolution, volume mounts, `RuntimePatch`, nodeSelectors, tolerations, and `torchrun` command configuration) is encapsulated inside `tpu_trainer.py`. Import `submit_multihost_training()` directly from `tpu_trainer.py` and pass `pvc_name="shared-workspace-rwx"`:
- **`image`**: Container image containing PyTorch and TorchTPU.
- **`src_dir`**: Directory containing your modular library files (`"src"`).
- **`main_script`**: Training entrypoint script (`"train.py"`).
- **`num_nodes`**: Number of TPU host nodes in the slice (`2`).
- **`tpus_per_node`**: Physical TPU chips per host (`4`).
- **`compute_class`**: GKE TPU ComputeClass (`"tpu-v5-8-multi-host"`).
- **`pvc_name`**: Shared `ReadWriteMany` PVC name (`"shared-workspace-rwx"`). Automatically resolves the isolated workspace `subPath` (`workspaces/<workspace-name>/.../src`) and mounts it at `/workspace` on all TPU hosts without creating a `ConfigMap`.

In [ ]:
from tpu_trainer import submit_multihost_training

# TPU training container image with PyTorch and torch_tpu
image = os.environ.get("TORCH_TPU_IMAGE") or f"{os.environ.get('REGISTRY', '').rstrip('/')}/jupyterlab:latest-tpu"

# Submit the distributed training job mounting the shared ReadWriteMany PVC
job_name = submit_multihost_training(
    image=image,
    src_dir="src",
    main_script="train.py",
    num_nodes=2,
    tpus_per_node=4,
    compute_class="tpu-v5-8-multi-host",
    pvc_name=shared_pvc_name,
    shared_mount_path=str(shared_mount_path),
)

## 4. Monitor TrainJob & Active Host Pods

Watch the `TrainJob` and inspect the worker Pods scheduled across the TPU nodes. Once all host pods transition to `Running`, distributed training begins.

In [ ]:
from tpu_trainer import wait_for_job_pods

# Wait until all host pods are active and Running
wait_for_job_pods(job_name, num_nodes=2)

## 5. Stream Distributed Logs Across TPU Hosts

Stream output from all active worker pods. Observe how each host initializes its local TorchTPU environment, exchanges SliceBuilder addresses, launches its 4 local TPU processes via `torchrun`, and executes distributed training across all 8 cores.

In [ ]:
from tpu_trainer import stream_job_logs

# Stream and filter distributed training logs across all TPU worker nodes
stream_job_logs(job_name, num_nodes=2)

## 6. Cleanup Resources

When you have finished inspecting the distributed training logs, execute this cell to delete the `TrainJob`. The shared `ReadWriteMany` volume (`shared-workspace-rwx`) remains intact so your code and saved artifacts persist across runs.

In [ ]:
from tpu_trainer import delete_training_job

# Clean up the TrainJob (the shared RWX PVC persists across runs)
delete_training_job(job_name, delete_configmap=False)